In [1]:
from google import genai
import os
import chromadb
from dotenv import load_dotenv, find_dotenv
load_dotenv()
api_key=os.getenv("GEMINI_API_KEY")
client=genai.Client(api_key=api_key)

In [ ]:
from pypdf import PdfReader
reader=PdfReader("CFBP Udaap.pdf")
document_text=""
for page in reader.pages:
    document_text+=page.extract_text()
print(document_text[:1000])

In [3]:
chat=client.chats.create(model="gemini-2.5-flash",
                         config={"system_instruction":f"""You are a senior compliance officer with 10 years 
                                of experience in banking regulations, credit risk policy, and internal controls.
                                
                                You have been given the following policy document to answer questions from:
                                
                                ========================
                                {document_text}
                                ========================
                                
                                Rules:
                                - Only answer from the document above
                                - Never make up information not in the document
                                - If the answer is not in the document, say "This information is not available in the provided document"
                                - Always respond in this format:
                                
                                Policy Area: [area]
                                Answer: [your answer]
                                Source: [quote the relevant section from the document]
                                Confidence: [High/Medium/Low]
                                """,
                                "temperature": 0.2,
                                "max_output_tokens": 1024})

In [4]:
while True:
    user_input=input("User : ")
    if user_input.lower() in ("end", "quit", "bye", "goodbye"):
        break
    response=chat.send_message(user_input)
    print( f"Model : {response.text}\n")    

In [ ]:
for m in client.models.list():
    if "generateContent" in m.supported_actions:
        print(m.name)

In [ ]:
def chunk_text(text, chunk_size=800, overlap=100):
    """
    Splits text into overlapping chunks.
    chunk_size: max characters per chunk
    overlap: characters shared between consecutive chunks (keeps context from being cut mid-sentence)
    """
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append(chunk)
        start += chunk_size - overlap  # move forward, but overlap with previous chunk
    return chunks

chunks = chunk_text(document_text)
print(f"Number of chunks: {len(chunks)}")
print("--- First chunk ---")
print(chunks[0])
print("--- Second chunk ---")
print(chunks[1])
print(len(chunks))

In [ ]:
def check_chunk_token_counts_gemini_embedding(chunks, client, model="gemini-embedding-001"):
    """
    Checks the token count of each chunk to ensure none exceed the embedding model's input limit before sending them for embedding.
    """
    max_tokens = 0
    over_limit = []
    for i, chunk in enumerate(chunks):
        result = client.models.count_tokens(model=model, contents=chunk)
        token_count = result.total_tokens
        max_tokens = max(max_tokens, token_count)
        if token_count > 2048:  # gemini-embedding-001's input limit
            over_limit.append((i, token_count))

    print(f"Largest chunk: {max_tokens} tokens")
    if over_limit:
        print(f"Chunks exceeding limit: {over_limit}")
    else:
        print("All chunks are within the embedding model's token limit.")
    return max_tokens, over_limit

max_tokens, over_limit = check_chunk_token_counts_gemini_embedding(chunks, client)

In [ ]:
def embed_chunks(chunks, client, model="gemini-embedding-001"):
    """
    Generates an embedding vector for each text chunk using Gemini's embedding model.
    Returns a list of vectors, same order as input chunks.
    """
    embeddings = []
    for chunk in chunks:
        result = client.models.embed_content(
            model=model,
            contents=chunk
        )
        embeddings.append(result.embeddings[0].values)
    return embeddings

embeddings = embed_chunks(chunks, client)
print(f"Number of embeddings: {len(embeddings)}")
print(f"Vector length (dimensions): {len(embeddings[0])}")
print(f"First 5 values of first vector: {embeddings[0][:5]}")

In [ ]:
# Sanity checks
print(len(chunks))              # just the number of chunks (e.g., 63)
print(chunks[0])                # just the first chunk's full text, cleanly
print(chunks[0][:200])          # just first 200 characters of the first chunk — quick peek
for i, c in enumerate(chunks[:5]):    # print first 5 chunks, each labeled
    print(f"--- Chunk {i} ({len(c)} chars) ---")
    print(c[:150], "...\n")

print("Length of chunks: ", len(chunks))
print(sum(1 for c in chunks if len(c.strip()) == 0))
print(1 if len(chunks) == len(embeddings) else 0)
print(len(set(len(e) for e in embeddings)) == 1)
print(all(v != 0 for v in embeddings[0][:10])) 
print(list(len(e) for e in embeddings))